### 1 - Configuration

In [0]:
# ============================================================
# ECCC Weather → Bronze Auto Loader ingestion
# Source: CSV files in /Volumes/ogp_dev/landing/zone/eccc/weather/
# Target: ogp_dev.bronze.eccc_weather_raw (Delta table)
# ============================================================

import uuid

CATALOG = "ogp_dev"
SCHEMA  = "bronze"
TABLE   = "eccc_weather_raw"
TABLE_FQN = f"{CATALOG}.{SCHEMA}.{TABLE}"

LANDING_PATH    = "/Volumes/ogp_dev/landing/zone/eccc/weather"
CHECKPOINT_PATH = f"/Volumes/ogp_dev/ops/checkpoints/bronze/{TABLE}"
SCHEMA_PATH     = f"{CHECKPOINT_PATH}/_schema"
RUN_ID          = f"manual_{uuid.uuid4()}"

print(f"Source path:      {LANDING_PATH}")
print(f"Target table:     {TABLE_FQN}")
print(f"Checkpoint path:  {CHECKPOINT_PATH}")
print(f"Run ID:           {RUN_ID}")

Source path:      /Volumes/ogp_dev/landing/zone/eccc/weather
Target table:     ogp_dev.bronze.eccc_weather_raw
Checkpoint path:  /Volumes/ogp_dev/ops/checkpoints/bronze/eccc_weather_raw
Run ID:           manual_7a9c737e-b97e-4a44-baf0-a8a0c8625de7


### 2 - Auto loader read stream

In [0]:
# Schema hints for ECCC columns that misinfer or vary across months:
#   - Precip Amount: comes as STRING when all NULL (e.g., dry July) — force DOUBLE
#   - Climate ID:   keep as STRING to preserve any leading zeros / non-numeric IDs
#   - Date/Time:    lock as TIMESTAMP
#   - Hmdx, Wind Chill: often inferred wrong when sparse — force INT
SCHEMA_HINTS = """
  `Climate ID`           STRING,
  `Date/Time (LST)`      TIMESTAMP,
  `Precip. Amount (mm)`  DOUBLE,
  `Hmdx`                 INT,
  `Wind Chill`           INT
"""

raw_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format",           "csv")
        .option("cloudFiles.schemaLocation",   SCHEMA_PATH)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaHints",      SCHEMA_HINTS)
        .option("header",                      "true")
        .option("encoding",                    "UTF-8")
        .load(LANDING_PATH)
)

print("Auto Loader schema:")
raw_stream.printSchema()

Auto Loader schema:
root
 |-- Longitude (x): double (nullable = true)
 |-- Latitude (y): double (nullable = true)
 |-- Station Name: string (nullable = true)
 |-- Climate ID: string (nullable = true)
 |-- Date/Time (LST): timestamp (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Time (LST): timestamp (nullable = true)
 |-- Flag: string (nullable = true)
 |-- Temp (°C): double (nullable = true)
 |-- Temp Flag: string (nullable = true)
 |-- Dew Point Temp (°C): double (nullable = true)
 |-- Dew Point Temp Flag: string (nullable = true)
 |-- Rel Hum (%): integer (nullable = true)
 |-- Rel Hum Flag: string (nullable = true)
 |-- Precip. Amount (mm): double (nullable = true)
 |-- Precip. Amount Flag: string (nullable = true)
 |-- Wind Dir (10s deg): integer (nullable = true)
 |-- Wind Dir Flag: string (nullable = true)
 |-- Wind Spd (km/h): integer (nullable = true)
 |-- Wind Spd Flag: string (nullable = t

### 3 - Transform: rename, cast, drop, enrich

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, expr

bronze_df = raw_stream.select(
    col("`Longitude (x)`").cast("double").alias("longitude"),
    col("`Latitude (y)`").cast("double").alias("latitude"),
    col("`Station Name`").cast("string").alias("station_name"),
    col("`Climate ID`").cast("string").alias("climate_id"),
    col("`Date/Time (LST)`").cast("timestamp").alias("event_ts_lst"),
    col("`Year`").cast("int").alias("year"),
    col("`Month`").cast("int").alias("month"),
    col("`Day`").cast("int").alias("day"),
    col("`Flag`").cast("string").alias("data_flag"),
    col("`Temp (°C)`").cast("double").alias("temp_c"),
    col("`Temp Flag`").cast("string").alias("temp_flag"),
    col("`Dew Point Temp (°C)`").cast("double").alias("dew_point_c"),
    col("`Dew Point Temp Flag`").cast("string").alias("dew_point_flag"),
    col("`Rel Hum (%)`").cast("int").alias("rel_humidity_pct"),
    col("`Rel Hum Flag`").cast("string").alias("rel_humidity_flag"),
    col("`Precip. Amount (mm)`").cast("double").alias("precip_mm"),
    col("`Precip. Amount Flag`").cast("string").alias("precip_flag"),
    col("`Wind Dir (10s deg)`").cast("int").alias("wind_dir_10s_deg"),
    col("`Wind Dir Flag`").cast("string").alias("wind_dir_flag"),
    col("`Wind Spd (km/h)`").cast("int").alias("wind_speed_kmh"),
    col("`Wind Spd Flag`").cast("string").alias("wind_speed_flag"),
    col("`Visibility (km)`").cast("double").alias("visibility_km"),
    col("`Visibility Flag`").cast("string").alias("visibility_flag"),
    col("`Stn Press (kPa)`").cast("double").alias("stn_pressure_kpa"),
    col("`Stn Press Flag`").cast("string").alias("stn_pressure_flag"),
    col("`Hmdx`").cast("int").alias("humidex"),
    col("`Hmdx Flag`").cast("string").alias("humidex_flag"),
    col("`Wind Chill`").cast("int").alias("wind_chill"),
    col("`Wind Chill Flag`").cast("string").alias("wind_chill_flag"),
    col("`Weather`").cast("string").alias("weather_desc"),
    # Keep _metadata accessible for the audit columns added below
    col("_metadata.file_path").alias("_source_file"),
)

bronze_df = (bronze_df
    .withColumn("_ingest_ts",     current_timestamp())
    .withColumn("_ingest_run_id", lit(RUN_ID))
)

print("Bronze schema (after enrichment):")
bronze_df.printSchema()

Bronze schema (after enrichment):
root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- station_name: string (nullable = true)
 |-- climate_id: string (nullable = true)
 |-- event_ts_lst: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- data_flag: string (nullable = true)
 |-- temp_c: double (nullable = true)
 |-- temp_flag: string (nullable = true)
 |-- dew_point_c: double (nullable = true)
 |-- dew_point_flag: string (nullable = true)
 |-- rel_humidity_pct: integer (nullable = true)
 |-- rel_humidity_flag: string (nullable = true)
 |-- precip_mm: double (nullable = true)
 |-- precip_flag: string (nullable = true)
 |-- wind_dir_10s_deg: integer (nullable = true)
 |-- wind_dir_flag: string (nullable = true)
 |-- wind_speed_kmh: integer (nullable = true)
 |-- wind_speed_flag: string (nullable = true)
 |-- visibility_km: double (nullable = true)
 |-- visibil

### 4 - Write to Delta

In [0]:
query = (
    bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(TABLE_FQN)
)

query.awaitTermination()
print(f"Ingestion complete — table: {TABLE_FQN}")

Ingestion complete — table: ogp_dev.bronze.eccc_weather_raw


### 5 - Verify

In [0]:
from pyspark.sql.functions import count, min as spark_min, max as spark_max

# Row count
print(f"Rows in {TABLE_FQN}: {spark.table(TABLE_FQN).count()}")

# Date range
print("\nTime range of ingested data:")
display(spark.table(TABLE_FQN).select(
    spark_min("event_ts_lst").alias("min_ts"),
    spark_max("event_ts_lst").alias("max_ts")
))

# Sample
print("\nFirst 5 rows (key columns):")
display(spark.table(TABLE_FQN).select(
    "event_ts_lst", "station_name", "climate_id",
    "temp_c", "rel_humidity_pct", "wind_speed_kmh", "weather_desc"
).orderBy("event_ts_lst").limit(5))

# Audit
print("\nFiles ingested:")
display(
    spark.table(TABLE_FQN)
        .groupBy("_source_file", "_ingest_run_id")
        .agg(count("*").alias("rows"))
)

Rows in ogp_dev.bronze.eccc_weather_raw: 744

Time range of ingested data:


min_ts,max_ts
2024-07-01T00:00:00.000Z,2024-07-31T23:00:00.000Z



First 5 rows (key columns):


event_ts_lst,station_name,climate_id,temp_c,rel_humidity_pct,wind_speed_kmh,weather_desc
2024-07-01T00:00:00.000Z,TORONTO INTL A,6158731,13.8,78,19,NA
2024-07-01T01:00:00.000Z,TORONTO INTL A,6158731,13.0,82,15,Clear
2024-07-01T02:00:00.000Z,TORONTO INTL A,6158731,12.1,86,12,NA
2024-07-01T03:00:00.000Z,TORONTO INTL A,6158731,11.1,92,12,NA
2024-07-01T04:00:00.000Z,TORONTO INTL A,6158731,11.7,87,14,Clear



Files ingested:


_source_file,_ingest_run_id,rows
/Volumes/ogp_dev/landing/zone/eccc/weather/eccc_weather_51459_202407.csv,manual_7a9c737e-b97e-4a44-baf0-a8a0c8625de7,744
